In [1]:
!pip install yt-dlp pydub tqdm


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the 'c:\program files\python38\python.exe -m pip install --upgrade pip' command.


In [2]:
import os
import yt_dlp
import math
from glob import glob
from pydub import AudioSegment
from pydub.utils import which

# ✅ FFmpeg Setup
FFMPEG_EXE_PATH = r"C:\Users\gaurav\Desktop\ffmpeg.exe"
os.environ["PATH"] += os.pathsep + os.path.dirname(FFMPEG_EXE_PATH)
AudioSegment.converter = FFMPEG_EXE_PATH
print("🔧 FFmpeg path detected by Pydub:", which("ffmpeg"))

# ✅ Refined Queries for Baby Choking / Uncomfortable Sounds
CLASSES = {
    "people-celebrating": [
        "people cheering and clapping sound",
        "crowd celebration sound effect",
        "group of people celebrating audio",
        "party celebration with cheering",
        "birthday celebration sound effect",
        "wedding crowd cheering sound",
        "people shouting in excitement",
        "team victory celebration audio",
        "congratulatory clapping and cheering",
        "group of friends celebrating sound"
    ]
}








CLIPS_PER_CLASS = 2000
CLIP_DURATION_MS = 15 * 1000
BATCH_SIZE = 500
TEMP_DIR = "temp_audio_files"
os.makedirs(TEMP_DIR, exist_ok=True)

# ✅ Get batch folder path
def get_batch_folder(class_name, index):
    batch_num = index // BATCH_SIZE + 1
    return f"{class_name}_batch_{batch_num}"

# ✅ Download + Slice into 15s audio clips
def download_and_split_audio(class_name, search_query):
    total_clips = sum(len(glob(os.path.join(f"{class_name}_batch_{i}", "*.wav")))
                      for i in range(1, math.ceil(CLIPS_PER_CLASS / BATCH_SIZE) + 1))

    if total_clips >= CLIPS_PER_CLASS:
        print(f"✅ {class_name}: Already has {total_clips} clips.")
        return

    temp_wav_path = os.path.join(TEMP_DIR, f"{class_name}_temp.wav")

    ydl_opts = {
        'format': 'bestaudio/best',
        'quiet': True,
        'outtmpl': os.path.join(TEMP_DIR, f"{class_name}_temp.%(ext)s"),
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'ffmpeg_location': FFMPEG_EXE_PATH,
        'noplaylist': True,
        'default_search': 'ytsearch20',
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        print(f"🔍 [{class_name}] Searching YouTube for: {search_query}")
        info = ydl.extract_info(search_query, download=False)
        entries = info.get('entries', [])

        audio_file_counter = total_clips + 1

        for entry in entries:
            if total_clips >= CLIPS_PER_CLASS:
                break

            try:
                url = entry['webpage_url']
                print(f"⬇ [{class_name}] Downloading from: {entry['title']}")
                ydl.download([url])

                audio = AudioSegment.from_wav(temp_wav_path)

                for i in range(0, len(audio), CLIP_DURATION_MS):
                    if total_clips >= CLIPS_PER_CLASS:
                        break

                    chunk = audio[i:i + CLIP_DURATION_MS]
                    if len(chunk) < CLIP_DURATION_MS:
                        continue

                    batch_folder = get_batch_folder(class_name, total_clips)
                    os.makedirs(batch_folder, exist_ok=True)

                    clip_name = f"{class_name}_audio_file{audio_file_counter}.wav"
                    clip_path = os.path.join(batch_folder, clip_name)
                    chunk.export(clip_path, format="wav")
                    print(f"✅ [{class_name}] Saved: {clip_name} ({total_clips+1}/{CLIPS_PER_CLASS})")

                    total_clips += 1
                    audio_file_counter += 1

                if os.path.exists(temp_wav_path):
                    os.remove(temp_wav_path)

            except Exception as e:
                print(f"⚠ [{class_name}] Error processing video: {e}")

# ✅ Process all search queries
for class_name, queries in CLASSES.items():
    for query in queries:
        download_and_split_audio(class_name, query)

# ✅ Final summary
print("\n📊 Final Clip Counts:")
for class_name in CLASSES:
    batches = glob(f"{class_name}_batch_*")
    total = sum(len(glob(os.path.join(b, "*.wav"))) for b in batches)
    print(f"{class_name}: {total} clips across {len(batches)} batches")


C:\Users\gaurav\AppData\Roaming\Python\Python310\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


🔧 FFmpeg path detected by Pydub: C:\Users\gaurav\Desktop\ffmpeg.exe
🔍 [people-celebrating] Searching YouTube for: people cheering and clapping sound
⬇ [people-celebrating] Downloading from: Big Crowd Cheer & Applause Clap Sound Effect
✅ [people-celebrating] Saved: people-celebrating_audio_file1.wav (1/2000)
✅ [people-celebrating] Saved: people-celebrating_audio_file2.wav (2/2000)
✅ [people-celebrating] Saved: people-celebrating_audio_file3.wav (3/2000)
✅ [people-celebrating] Saved: people-celebrating_audio_file4.wav (4/2000)
✅ [people-celebrating] Saved: people-celebrating_audio_file5.wav (5/2000)
⬇ [people-celebrating] Downloading from: Applause Crowd Cheering sound effect
✅ [people-celebrating] Saved: people-celebrating_audio_file6.wav (6/2000)
⬇ [people-celebrating] Downloading from: Stadium Crowd Sound Effects | One Hour | HQ
✅ [people-celebrating] Saved: people-celebrating_audio_file7.wav (7/2000)
✅ [people-celebrating] Saved: people-celebrating_audio_file8.wav (8/2000)
✅ [people-